In [8]:
import os
import pandas as pd
import os
os.getcwd()

search_dir  =  os.path.join(os.path.abspath(os.path.join(os.getcwd(), os.pardir)), "logs")

print("Current working directory:", os.getcwd())
print("Search directory:", search_dir)

Current working directory: c:\Users\ritux\OneDrive - Danmarks Tekniske Universitet\Dokumenter\Esbjerg\4 - Models\GitHubRepo\notebooks
Search directory: c:\Users\ritux\OneDrive - Danmarks Tekniske Universitet\Dokumenter\Esbjerg\4 - Models\GitHubRepo\logs


### New Approach, search for file names

In [9]:
# ── Config ────────────────────────────────────────────────────────────────────

objectives = ["Econ", "QoL"]

# datasets per objective
datasets = {
    "Econ": ["econ_combined"] ,
    "QoL":  ["qol_combined"],
}

# single-head components
components = {
    "Econ": ["delay", "cancel", "infra"],
    "QoL":  ["qol", "action", "maintenance"],
}

# profiles (== PARAMETERS) per model family
profiles = {
    "MLP":  ["simple_mlp", "deep_mlp", "wide_mlp", "large_mlp"],
    "Auto": ["small_bottleneck", "deep_bottleneck", "medium_bottleneck",
             "wide_bottleneck", "wide_xl_bottleneck", "large_bottleneck"],
    "VAE":  ["small_vae", "medium_vae", "wide_vae", "large_vae"],
    "GP":   ["tiny_gp", "small_gp", "medium_gp", "large_gp","xlarge_gp"], 
    "XGB":  ["small_xgb", "medium_xgb", "large_xgb"],
    "GNN": ["small_gnn", "medium_gnn", "deep_gnn","sage_gnn"]
}


Modes = ["single", "multi"]
Loader_Modes = ["chunked", "whole"]

ALL_COMPONENTS = ["delay", "cancel", "infra", "qol", "action", "maintenance"]



### Single

In [10]:
# ── Search ────────────────────────────────────────────────────────────────────

# rows keyed by (Model, Parameter) → dict of component → status string
results_single: dict[tuple, dict[str, str]] = {}

for objective in objectives:
    for dataset in datasets[objective]:
        for model, model_profiles in profiles.items():
            for parameter in model_profiles:
                for component in components[objective]:
                    found = False 

                    # train_qol_combined_single_QoL_qol_GNN_small_gnn_61618564
                    filename_pattern = f"{dataset}_single_{objective}_{component}_{model}_{parameter}_"
                    print(f"\nSearching for files with pattern: {filename_pattern}")

                    # Find files in the search_dir that match the filename pattern
                    matching_files = [
                        os.path.join(root, fname)
                        for root, dirs, files in os.walk(search_dir, topdown=False)
                        for fname in files
                        if fname.endswith(".out") and filename_pattern in fname
                    ]
                    print(f"Found {len(matching_files)} matching file(s):")

                    model_call = (
                        f"OBJECTIVE={objective} DATASET={dataset} MODE=single "
                        f"MODEL={model} PARAMETERS={parameter} COMPONENT={component} "
                        f"LOADER_MODE=chunked"
                    )
                    terms = [model_call, "Starting surrogate training"]
                    # print(f"Searching for: {terms}")

                    # Sort matching files by job ID extracted from the filename 
                    matching_files.sort(key=lambda f: int(f.split("_")[-1].split(".")[0]), reverse=True)

                    for fpath in matching_files:
                        print(f"Checking file: {fpath}")
                        if not fpath.endswith(".out"):
                            continue
                        try:
                            text = open(fpath, encoding="utf-8", errors="ignore").read()
                            if not all(t in text for t in terms):
                                continue


                            found = True
                            print(f"  Match: {fpath}")

                            if "Finished successfully:" in text:
                                status_str = "x"
                                print("  → Success")
                                break   # first successful matching file wins
                            elif "walltime" in text:
                                status_str = "walltime"
                                print("  → Walltime exceeded")
                            elif "Unable to allocate " in text:
                                status_str = "OOM"
                                print("  → Memory allocation error")
                            else:
                                status_str = "failed"
                                print("  → Failed (unknown reason)")
                                # break

                            
                        except Exception as e:
                            print(f"  Could not read {fpath}: {e}")


                    if not found:
                        status_str = ""   # no .out file at all

                    key = (model, parameter)
                    if key not in results_single:
                        results_single[key] = {}
                    if results_single[key].get(component) == "x":
                        continue  # don't overwrite a success with a failure
                    else:
                        # overwrite with the latest status if it is failed
                        if results_single[key].get(component) == "failed":
                            results_single[key][component] = status_str
                        else:
                            results_single[key][component] = status_str

                            
# Optional: save to CSV
# df.to_csv("surrogate_status.csv", index=False)


Searching for files with pattern: econ_combined_single_Econ_delay_MLP_simple_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_cancel_MLP_simple_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_infra_MLP_simple_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_delay_MLP_deep_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_cancel_MLP_deep_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_infra_MLP_deep_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_delay_MLP_wide_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_cancel_MLP_wide_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_single_Econ_infra_MLP_wide_mlp_
Found 0 matching file(s):

Searching for files with pa

In [11]:
# ── Build table ───────────────────────────────────────────────────────────────

rows = []
for (model, parameter), comp_dict in results_single.items():
    row = {"Model": model, "Parameter":  " ".join(parameter.split("_")[:-1])}
    for comp in ALL_COMPONENTS:
        row[comp] = comp_dict.get(comp, "—")   # "—" = not applicable
    rows.append(row)

df_single = pd.DataFrame(rows, columns=["Model", "Parameter"] + ALL_COMPONENTS)
# df_single = df_single.sort_values(["Model", "Parameter"]).reset_index(drop=True)

print("\n")
print(df_single.to_string(index=False))




Model Parameter delay cancel infra qol action maintenance
  MLP    simple                                          
  MLP      deep                                          
  MLP      wide                                          
  MLP     large                                          
 Auto     small                                          
 Auto      deep                                          
 Auto    medium                                          
 Auto      wide                                          
 Auto   wide xl                                          
 Auto     large                                          
  VAE     small                                          
  VAE    medium                                          
  VAE      wide                                          
  VAE     large                                          
   GP      tiny                                          
   GP     small                                          
   GP    med

In [12]:
latex =df_single.to_latex(index=False)
latex = latex.replace("✓", "x")
latex = latex.replace("toprule", "hline")
latex = latex.replace("midrule", "hline")
latex = latex.replace("bottomrule", "hline")

print(latex)

\begin{tabular}{llllllll}
\hline
Model & Parameter & delay & cancel & infra & qol & action & maintenance \\
\hline
MLP & simple &  &  &  &  &  &  \\
MLP & deep &  &  &  &  &  &  \\
MLP & wide &  &  &  &  &  &  \\
MLP & large &  &  &  &  &  &  \\
Auto & small &  &  &  &  &  &  \\
Auto & deep &  &  &  &  &  &  \\
Auto & medium &  &  &  &  &  &  \\
Auto & wide &  &  &  &  &  &  \\
Auto & wide xl &  &  &  &  &  &  \\
Auto & large &  &  &  &  &  &  \\
VAE & small &  &  &  &  &  &  \\
VAE & medium &  &  &  &  &  &  \\
VAE & wide &  &  &  &  &  &  \\
VAE & large &  &  &  &  &  &  \\
GP & tiny &  &  &  &  &  &  \\
GP & small &  &  &  &  &  &  \\
GP & medium &  &  &  &  &  &  \\
GP & large &  &  &  &  &  &  \\
GP & xlarge &  &  &  &  &  &  \\
XGB & small &  &  &  &  &  &  \\
XGB & medium &  &  &  &  &  &  \\
XGB & large &  &  &  &  &  &  \\
GNN & small &  &  &  &  &  &  \\
GNN & medium &  &  &  &  &  &  \\
GNN & deep &  &  &  &  &  &  \\
GNN & sage &  &  &  &  &  &  \\
\hline
\end{tabular}



#### Multi

In [13]:
# ── Search ────────────────────────────────────────────────────────────────────

# rows keyed by (Model, Parameter) → dict of component → status string
results_multi: dict[tuple, dict[str, str]] = {}

for objective in objectives:
    for dataset in datasets[objective]:
        for model, model_profiles in profiles.items():
            for parameter in model_profiles:
                found = False 

                # train_qol_combined_multi_QoL_all_VAE_medium_vae_61661536
                filename_pattern = f"{dataset}_multi_{objective}_all_{model}_{parameter}_"
                print(f"\nSearching for files with pattern: {filename_pattern}")

                # Find files in the search_dir that match the filename pattern
                matching_files = [
                    os.path.join(root, fname)
                    for root, dirs, files in os.walk(search_dir, topdown=False)
                    for fname in files
                    if fname.endswith(".out") and filename_pattern in fname
                ]
                print(f"Found {len(matching_files)} matching file(s):")

                model_call = (
                    f"OBJECTIVE={objective} DATASET={dataset} MODE=multi "
                    f"MODEL={model} PARAMETERS={parameter} COMPONENT=all "
                    f"LOADER_MODE=chunked"
                )
                terms = [model_call, "Starting surrogate training"]
                # print(f"Searching for: {terms}")

                # Sort matching files by job ID extracted from the filename 
                matching_files.sort(key=lambda f: int(f.split("_")[-1].split(".")[0]), reverse=True)

                for fpath in matching_files:
                    print(f"Checking file: {fpath}")
                    if not fpath.endswith(".out"):
                        continue
                    try:
                        text = open(fpath, encoding="utf-8", errors="ignore").read()
                        if not all(t in text for t in terms):
                            continue

                        job_id = fpath.split("_")[-1].split(".")[0]
                        found = True
                        print(f"  Match: job_id = {job_id}")

                        if "Finished successfully:" in text:
                            status_str = "✓"
                            print("  → Success")
                            break   # first successful matching file wins
                        elif "walltime" in text:
                            status_str = "walltime"
                            print("  → Walltime exceeded")                                
                        elif "Unable to allocate " in text:
                            status_str = "OOM"
                            print("  → Memory allocation error")
                        else:
                            status_str = "failed"
                            print("  → Failed (unknown reason)")
                            # break   # first matching file wins, even if failed

                    except Exception as e:
                        print(f"  Could not read {fpath}: {e}")


                if not found:
                    status_str = ""   # no .out file at all

                key = (model, parameter)
                if key not in results_multi:
                    results_multi[key] = {}
                if results_multi[key].get(objective) == "✓":
                    continue  # don't overwrite a success with a failure
                else:
                    # overwrite with the latest status if it is failed
                    if results_multi[key].get(objective) == "failed":
                        results_multi[key][objective] = status_str
                    else:
                        results_multi[key][objective] = status_str

                            
# Optional: save to CSV
# df.to_csv("surrogate_status.csv", index=False)


Searching for files with pattern: econ_combined_multi_Econ_all_MLP_simple_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_MLP_deep_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_MLP_wide_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_MLP_large_mlp_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_Auto_small_bottleneck_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_Auto_deep_bottleneck_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_Auto_medium_bottleneck_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_Auto_wide_bottleneck_
Found 0 matching file(s):

Searching for files with pattern: econ_combined_multi_Econ_all_Auto_wide_xl_bottleneck_
Found 0 matching file(s):

Searching for 

In [14]:
# ── Build table ───────────────────────────────────────────────────────────────


rows = []
for model, model_profiles in profiles.items():
    for parameter in model_profiles:
        row = {"Model": model, "Parameter":  " ".join(parameter.split("_")[:-1])}
        for obj in objectives:
            row[obj] = results_multi.get((model, parameter), {}).get(obj, "—")
        rows.append(row)



df_multi = pd.DataFrame(rows, columns=["Model", "Parameter"] + objectives)

print("\n")
print(df_multi.to_string(index=False))




Model Parameter Econ QoL
  MLP    simple         
  MLP      deep         
  MLP      wide         
  MLP     large         
 Auto     small         
 Auto      deep         
 Auto    medium         
 Auto      wide         
 Auto   wide xl         
 Auto     large         
  VAE     small         
  VAE    medium         
  VAE      wide         
  VAE     large         
   GP      tiny         
   GP     small         
   GP    medium         
   GP     large         
   GP    xlarge         
  XGB     small         
  XGB    medium         
  XGB     large         
  GNN     small         
  GNN    medium         
  GNN      deep         
  GNN      sage         
